# Behavior decoding judge for BSS-cleaned calcium traces

This notebook tests whether BSS-cleaned extracted traces preserve behavior-decodable structure from the raw extracted traces. The key comparisons are within-version decoding and transfer decoding between raw and cleaned trace representations.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

candidates = [Path.cwd(), *Path.cwd().parents]
PROJECT_ROOT = next(p for p in candidates if (p / "pyproject.toml").exists() and (p / "src").exists())
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from behavior_decoding import (
    load_trace_variants,
    make_behavior_targets,
    run_decoding_experiment,
    save_result,
)

plt.rcParams.update({"figure.figsize": (10, 4), "axes.spines.top": False, "axes.spines.right": False})
PROJECT_ROOT

## Configuration

In [ ]:
DATASET_DIR = PROJECT_ROOT / "data" / "v2a-RSNs" / "new_run2_844ROI"
TAIL_ANGLE_PATH = PROJECT_ROOT / "data" / "v2a-RSNs" / "220127_F4_run2_tail_angle.npy"
OUTPUT_DIR = PROJECT_ROOT / "outputs" / "behavior_decoding" / "new_run2_844ROI"

RAW_NAME = "fluo_signals_no_NaN.npy"
CLEANED_GLOB = "cleanedNew_*.npy"
LAGS = (0, 1, 2)
TARGET_SHIFT = 0
N_SPLITS = 5
GAP = 3
RIDGE_ALPHA = 10.0
BOUT_QUANTILE = 0.75
SMOOTH_WINDOW = 3

DATASET_DIR, TAIL_ANGLE_PATH, OUTPUT_DIR

## Load raw and cleaned traces

In [ ]:
variants = load_trace_variants(DATASET_DIR, raw_name=RAW_NAME, cleaned_glob=CLEANED_GLOB)
trace_table = pd.DataFrame(
    {"version": v.name, "path": str(v.path.relative_to(PROJECT_ROOT)), "shape": [v.traces.shape for v in variants]}
)
trace_table

In [ ]:
raw = variants[0].traces
example_cleaned = variants[1].traces if len(variants) > 1 else raw
neuron_idx = 0
time_slice = slice(0, min(600, raw.shape[0]))

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(raw[time_slice, neuron_idx], label="raw", lw=1.2)
ax.plot(example_cleaned[time_slice, neuron_idx], label=variants[1].name if len(variants) > 1 else "cleaned", lw=1.2)
ax.set_title(f"Example extracted trace, neuron {neuron_idx}")
ax.set_xlabel("calcium frame")
ax.set_ylabel("fluorescence")
ax.legend();

## Align tail behavior to calcium frames

In [ ]:
tail_angle = np.load(TAIL_ANGLE_PATH)
targets = make_behavior_targets(
    tail_angle,
    n_frames=raw.shape[0],
    bout_quantile=BOUT_QUANTILE,
    smooth_window=SMOOTH_WINDOW,
)

pd.Series(
    {
        "tail_samples": tail_angle.shape[0],
        "calcium_frames": raw.shape[0],
        "samples_per_calcium_frame": tail_angle.shape[0] / raw.shape[0],
        "bout_threshold": targets.bout_threshold,
        "bout_fraction": targets.bout_state.mean(),
    }
)

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(12, 7), sharex=True)
frames = np.arange(raw.shape[0])
axes[0].plot(frames, targets.angle, color="black", lw=0.8)
axes[0].set_ylabel("tail angle")
axes[1].plot(frames, targets.vigor, color="tab:blue", lw=0.9)
axes[1].axhline(targets.bout_threshold, color="tab:red", ls="--", lw=1)
axes[1].set_ylabel("tail vigor")
axes[2].plot(frames, targets.bout_state, color="tab:green", lw=0.9)
axes[2].set_ylabel("bout state")
axes[2].set_xlabel("calcium frame")
fig.suptitle("Tail behavior aligned to calcium frames");

## Run decoding experiments

In [ ]:
common_kwargs = dict(
    variants=variants,
    lags=LAGS,
    target_shift=TARGET_SHIFT,
    n_splits=N_SPLITS,
    gap=GAP,
    ridge_alpha=RIDGE_ALPHA,
    include_transfer=True,
    include_null=True,
)

vigor_result = run_decoding_experiment(
    target=targets.vigor,
    target_name="tail_vigor",
    task="regression",
    **common_kwargs,
)

bout_result = run_decoding_experiment(
    target=targets.bout_state,
    target_name="bout_state",
    task="classification",
    **common_kwargs,
)

metrics = pd.concat([vigor_result.metrics, bout_result.metrics], ignore_index=True)
summary = pd.concat([vigor_result.summary, bout_result.summary], ignore_index=True)
summary.head()

## Within-version decoding scores

In [ ]:
within_vigor = summary.query("target == 'tail_vigor' and comparison == 'within'").copy()
within_bout = summary.query("target == 'bout_state' and comparison == 'within'").copy()

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].bar(within_vigor["test_version"], within_vigor["r2_mean"], yerr=within_vigor["r2_std"], color="tab:blue", alpha=0.8)
axes[0].axhline(0, color="black", lw=0.8)
axes[0].set_title("Tail vigor decoding within each trace version")
axes[0].set_ylabel("CV R2")
axes[0].tick_params(axis="x", rotation=45)

axes[1].bar(within_bout["test_version"], within_bout["balanced_accuracy_mean"], yerr=within_bout["balanced_accuracy_std"], color="tab:green", alpha=0.8)
axes[1].axhline(0.5, color="black", lw=0.8, ls="--")
axes[1].set_title("Bout-state decoding within each trace version")
axes[1].set_ylabel("balanced accuracy")
axes[1].tick_params(axis="x", rotation=45)
plt.tight_layout();

## Transfer scores: raw-trained decoder tested on cleaned traces

In [ ]:
transfer_vigor = summary.query("target == 'tail_vigor' and comparison == 'transfer_raw_to_clean'").copy()
transfer_bout = summary.query("target == 'bout_state' and comparison == 'transfer_raw_to_clean'").copy()

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].bar(transfer_vigor["test_version"], transfer_vigor["r2_mean"], yerr=transfer_vigor["r2_std"], color="tab:orange", alpha=0.85)
axes[0].axhline(0, color="black", lw=0.8)
axes[0].set_title("Raw-trained tail-vigor decoder on cleaned traces")
axes[0].set_ylabel("transfer R2")
axes[0].tick_params(axis="x", rotation=45)

axes[1].bar(transfer_bout["test_version"], transfer_bout["balanced_accuracy_mean"], yerr=transfer_bout["balanced_accuracy_std"], color="tab:red", alpha=0.8)
axes[1].axhline(0.5, color="black", lw=0.8, ls="--")
axes[1].set_title("Raw-trained bout decoder on cleaned traces")
axes[1].set_ylabel("transfer balanced accuracy")
axes[1].tick_params(axis="x", rotation=45)
plt.tight_layout();

## Prediction traces for one fold

In [ ]:
predictions = pd.concat([vigor_result.predictions, bout_result.predictions], ignore_index=True)
example_version = variants[1].name if len(variants) > 1 else "raw"
fold = 0
plot_df = predictions.query(
    "target == 'tail_vigor' and fold == @fold and "
    "((comparison == 'within' and test_version == 'raw') or "
    "(comparison == 'transfer_raw_to_clean' and test_version == @example_version))"
).copy()

fig, ax = plt.subplots(figsize=(13, 4))
for label, group in plot_df.groupby(["comparison", "test_version"]):
    ax.plot(group["time_index"], group["y_pred"], lw=1, label=f"pred {label[0]}:{label[1]}")
truth = plot_df.drop_duplicates("time_index").sort_values("time_index")
ax.plot(truth["time_index"], truth["y_true"], color="black", lw=1.4, label="true tail vigor")
ax.set_title(f"Tail-vigor predictions, fold {fold}")
ax.set_xlabel("calcium frame")
ax.set_ylabel("tail vigor")
ax.legend();

## Save outputs

In [ ]:
save_result(vigor_result, OUTPUT_DIR, "tail_vigor")
save_result(bout_result, OUTPUT_DIR, "bout_state")
metrics.to_csv(OUTPUT_DIR / "all_fold_metrics.csv", index=False)
summary.to_csv(OUTPUT_DIR / "all_summary.csv", index=False)
predictions.to_csv(OUTPUT_DIR / "all_predictions.csv", index=False)
OUTPUT_DIR